# # voice
#
# > Speech to text on this machine, for think-aloud answers.

In [ ]:
#| default_exp voice

In [ ]:
from fastcore.test import *

In [ ]:
#| export
import os, tempfile, wave
from pathlib import Path
from fastcore.utils import *

# The browser records 16 kHz mono WAV and posts it. `transcribe` turns it into text on this machine.
#
# - `gemma` (default): Gemma 4 E2B on LiteRT through rishi. The model is in the Hugging Face cache already, and rishi keeps
#   the engine loaded between clips.
# - `whisper`: mlx-audio Whisper (`pip install abhyasa[whisper]`); `$ABHYASA_WHISPER` names the model.
#
# `$ABHYASA_STT` picks one.

In [ ]:
#| export
STT_SP = 'You transcribe speech exactly. Keep technical terms, Sanskrit words (in IAST) and maths as spoken. Reply with the transcript only.'
DFLT_WHISPER = 'mlx-community/whisper-large-v3-turbo'

def _gemma(path, hint=''):
    from rishi import Chat, resp_text
    from rishi.litert import gemma4_e2b
    chat = Chat(gemma4_e2b, sp=STT_SP)
    try: return resp_text(chat([f'Transcribe this clip.{" Words it may use: " + hint if hint else ""}', Path(path)]))
    finally: chat.close()

_whisper = {}
def _mlx_whisper(path, hint=''):
    from mlx_audio.stt.utils import load_model
    nm = os.getenv('ABHYASA_WHISPER', DFLT_WHISPER)
    if nm not in _whisper: _whisper[nm] = load_model(nm)
    return _whisper[nm].generate(str(path)).text

ENGINES = dict(gemma=_gemma, whisper=_mlx_whisper)

def transcribe(audio, engine:str=None, hint:str='') -> str:
    "Text for a WAV file or WAV bytes. `hint` lists words the speaker may use, such as the concept's name."
    eng = ENGINES[engine or os.getenv('ABHYASA_STT', 'gemma')]
    if isinstance(audio, (bytes, bytearray)):
        with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as f: f.write(audio)
        try: return eng(f.name, hint).strip()
        finally: os.unlink(f.name)
    return eng(audio, hint).strip()

In [ ]:
#| export
def wav_seconds(audio:bytes) -> float:
    "The length of WAV bytes, in seconds; 0 when they are not WAV."
    import io
    try:
        with wave.open(io.BytesIO(audio)) as w: return w.getnframes() / w.getframerate()
    except Exception: return 0.

In [ ]:
import io, struct
b = io.BytesIO()
with wave.open(b, 'wb') as w: w.setnchannels(1); w.setsampwidth(2); w.setframerate(16000); w.writeframes(b'\0\0' * 8000)
test_eq(wav_seconds(b.getvalue()), .5)
test_eq(wav_seconds(b'nope'), 0.)

# A real clip, with the model (slow; `nbdev-test --flags slow`):

In [ ]:
#| slow
clip = Path('../../rishi/nbs/speech.wav')
if clip.exists(): print(transcribe(clip))

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()